liður 4


In [1]:
import pandas as pd
import numpy as np

# =========================================================
# 1. GOOGLE SHEETS
# =========================================================

SHEET_ID = "1dld6f07WhVLpz-iDmhCejRfWrv8v5hDKFGPKrWkvFdI"
GID_STATIC = "0"
GID_CARRA  = "274008059"
GID_QOBS   = "1766120044"

def read_gsheet_csv(sheet_id, gid):
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    return pd.read_csv(url)

static = read_gsheet_csv(SHEET_ID, GID_STATIC)
carra  = read_gsheet_csv(SHEET_ID, GID_CARRA)
qobs   = read_gsheet_csv(SHEET_ID, GID_QOBS)

# =========================================================
# 2. HREINSA
# =========================================================

for df in [static, carra, qobs]:
    df.columns = [str(c).strip() for c in df.columns]
    df.dropna(axis=1, how="all", inplace=True)
    df.dropna(axis=0, how="all", inplace=True)

static = static.loc[:, ~static.columns.str.contains("^Unnamed", case=False, na=False)]
carra  = carra.loc[:, ~carra.columns.str.contains("^Unnamed", case=False, na=False)]
qobs   = qobs.loc[:, ~qobs.columns.str.contains("^Unnamed", case=False, na=False)]

# =========================================================
# 3. VELJA DÁLKA
# =========================================================

area_km2 = float(pd.to_numeric(static["area_calc"], errors="coerce").iloc[0])

carra = carra[["YYYY", "MM", "DD", "prec_carra", "total_et_carra"]].copy()
qobs_cols = ["YYYY", "MM", "DD", "qobs"]
if "qc_flag" in qobs.columns:
    qobs_cols.append("qc_flag")
qobs = qobs[qobs_cols].copy()

for col in ["YYYY", "MM", "DD", "prec_carra", "total_et_carra"]:
    carra[col] = pd.to_numeric(carra[col], errors="coerce")

for col in ["YYYY", "MM", "DD", "qobs"]:
    qobs[col] = pd.to_numeric(qobs[col], errors="coerce")

if "qc_flag" in qobs.columns:
    qobs["qc_flag"] = pd.to_numeric(qobs["qc_flag"], errors="coerce")

# no-data
nodata_vals = [-9999, -999, -99, -9.99e8]
carra = carra.replace(nodata_vals, np.nan)
qobs = qobs.replace(nodata_vals, np.nan)

# =========================================================
# 4. DAGSETNINGAR OG SAMEINING
# =========================================================

carra["date"] = pd.to_datetime(
    carra[["YYYY", "MM", "DD"]].rename(columns={"YYYY": "year", "MM": "month", "DD": "day"}),
    errors="coerce"
)
qobs["date"]  = pd.to_datetime(
    qobs[["YYYY", "MM", "DD"]].rename(columns={"YYYY": "year", "MM": "month", "DD": "day"}),
    errors="coerce"
)

carra = carra.dropna(subset=["date", "prec_carra", "total_et_carra"])
qobs = qobs.dropna(subset=["date", "qobs"])

df = pd.merge(carra, qobs, on=["YYYY", "MM", "DD", "date"], how="inner")

# =========================================================
# 5. GÆÐASÍA
# =========================================================

# Hér geturðu valið að halda bara ákveðnum qc_flag gildum
USE_QC_FILTER = False
VALID_QC_FLAGS = [40]

if USE_QC_FILTER and "qc_flag" in df.columns:
    df = df[df["qc_flag"].isin(VALID_QC_FLAGS)].copy()

# =========================================================
# 6. EININGAR
# =========================================================

# ef þú kemst að því að CARRA sé í metrum, breyttu í True
CONVERT_CARRA_FROM_M_TO_MM = False

if CONVERT_CARRA_FROM_M_TO_MM:
    df["prec_carra"] = df["prec_carra"] * 1000
    df["total_et_carra"] = df["total_et_carra"] * 1000

# qobs: m3/s -> mm/dag
df["q_mm_day"] = (df["qobs"] * 86.4) / area_km2

# dagleg breyting geymslu
df["deltaS_day"] = df["prec_carra"] - df["total_et_carra"] - df["q_mm_day"]

# =========================================================
# 7. TÍMABIL
# =========================================================

START_DATE = "1993-10-01"
END_DATE   = "2023-09-30"

df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()

# =========================================================
# 8. VATNAÁR
# =========================================================

# Okt-Sep vatnaár
df["water_year"] = np.where(df["MM"] >= 10, df["YYYY"] + 1, df["YYYY"])

# =========================================================
# 9. HEILDAR NIÐURSTÖÐUR
# =========================================================

P_total = df["prec_carra"].sum()
ET_total = df["total_et_carra"].sum()
Q_total = df["q_mm_day"].sum()
deltaS_total = df["deltaS_day"].sum()

area_m2 = area_km2 * 1_000_000
deltaS_total_m3 = (deltaS_total / 1000) * area_m2

print("================================")
print("HEILDARNIÐURSTÖÐUR")
print("================================")
print("Flatarmál (km²):", round(area_km2, 2))
print("Tímabil:", df["date"].min().date(), "til", df["date"].max().date())
print("Fjöldi daga:", len(df))
print(f"P samtals (mm):  {P_total:.2f}")
print(f"ET samtals (mm): {ET_total:.2f}")
print(f"Q samtals (mm):  {Q_total:.2f}")
print(f"ΔS samtals (mm): {deltaS_total:.2f}")
print(f"ΔS samtals (m3): {deltaS_total_m3:.2f}")

# =========================================================
# 10. ÁRLEGAR NIÐURSTÖÐUR
# =========================================================

annual = (
    df.groupby("water_year")[["prec_carra", "total_et_carra", "q_mm_day", "deltaS_day"]]
    .sum()
    .rename(columns={
        "prec_carra": "P_mm",
        "total_et_carra": "ET_mm",
        "q_mm_day": "Q_mm",
        "deltaS_day": "deltaS_mm"
    })
)

annual["deltaS_m3"] = (annual["deltaS_mm"] / 1000) * area_m2

print("\n================================")
print("ÁRLEGAR NIÐURSTÖÐUR")
print("================================")
print(annual.head())
print("\nLýsing á árlegri ΔS:")
print(annual["deltaS_mm"].describe())

# =========================================================
# 11. MEÐALTÖL
# =========================================================

mean_annual = annual.mean(numeric_only=True)

print("\n================================")
print("MEÐAL VATNAÁR")
print("================================")
print(f"Meðal P (mm/ár):  {mean_annual['P_mm']:.2f}")
print(f"Meðal ET (mm/ár): {mean_annual['ET_mm']:.2f}")
print(f"Meðal Q (mm/ár):  {mean_annual['Q_mm']:.2f}")
print(f"Meðal ΔS (mm/ár): {mean_annual['deltaS_mm']:.2f}")

# =========================================================
# 12. MÁNAÐARLEG NIÐURSTAÐA
# =========================================================

monthly = (
    df.set_index("date")[["prec_carra", "total_et_carra", "q_mm_day", "deltaS_day"]]
    .resample("ME")
    .sum()
)

print("\n================================")
print("FYRSTU MÁNUÐIR")
print("================================")
print(monthly.head())

# =========================================================
# 13. VISTA
# =========================================================

df.to_csv("deltaS_daily_recalc.csv", index=False)
annual.to_csv("deltaS_annual_recalc.csv")
monthly.to_csv("deltaS_monthly_recalc.csv")

print("\nVistað:")
print("- deltaS_daily_recalc.csv")
print("- deltaS_annual_recalc.csv")
print("- deltaS_monthly_recalc.csv")

HEILDARNIÐURSTÖÐUR
Flatarmál (km²): 390.56
Tímabil: 1993-10-01 til 2023-09-30
Fjöldi daga: 10957
P samtals (mm):  24521.79
ET samtals (mm): 5704.75
Q samtals (mm):  24616.82
ΔS samtals (mm): -5799.78
ΔS samtals (m3): -2265161753.60

ÁRLEGAR NIÐURSTÖÐUR
               P_mm    ET_mm        Q_mm   deltaS_mm     deltaS_m3
water_year                                                        
1994        723.057  191.634  830.743875 -299.320875 -1.169028e+08
1995        832.596  148.920  855.717493 -172.041493 -6.719253e+07
1996        744.786  201.809  733.961983 -190.984983 -7.459109e+07
1997        911.277  170.736  884.531503 -143.990503 -5.623693e+07
1998        766.119  182.127  844.021549 -260.029549 -1.015571e+08

Lýsing á árlegri ΔS:
count     30.000000
mean    -193.325972
std       71.615211
min     -367.216313
25%     -250.674804
50%     -177.779515
75%     -144.709699
max      -42.490982
Name: deltaS_mm, dtype: float64

MEÐAL VATNAÁR
Meðal P (mm/ár):  817.39
Meðal ET (mm/ár): 190.16